# RAIDS-NIDS v0.19 external dataset and guard comparison

Run this notebook from the `raids-nids` project root. The protocol is already frozen. Do not change episode families, guard candidates, window boundaries, or model seeds after viewing real outcomes.

In [ ]:
from pathlib import Path
import json
import itertools

import raids_nids
import river

from raids_nids.audit import audit_dataset
from raids_nids.config import deep_merge, load_yaml
from raids_nids.guard_benchmark import aggregate_guard_benchmarks, run_guard_benchmark
from raids_nids.unsw_events import build_unsw_event_suite, build_unsw_temporal_cache

print('raids-nids:', raids_nids.__version__)
print('river:', river.__version__)
assert raids_nids.__version__ == '0.1.9'
assert river.__version__ == '0.25.0'

In [ ]:
RAW = Path('data/raw/NF-UNSW-NB15-v3.csv')
CACHE = Path('data/derived/v019_unsw_temporal.npz')
EVENT_DIR = Path('data/derived/v019_unsw_events')
RESULTS_DIR = Path('results/v019_external_guard_comparison/runs')
AGGREGATE_DIR = Path('results/v019_external_guard_comparison/aggregate')

assert RAW.exists(), f'Place the official dataset at: {RAW.resolve()}'
print('Raw dataset:', RAW.resolve())
print('Size (GiB):', round(RAW.stat().st_size / 1024**3, 3))

## 1. Build the stable temporal cache

The expected row count is 2,365,424. Stop if the release does not match.

In [ ]:
if CACHE.exists() and CACHE.with_suffix('.json').exists():
    cache_report = json.loads(CACHE.with_suffix('.json').read_text(encoding='utf-8'))
    print('Using existing cache:', CACHE.resolve())
else:
    cache_report = build_unsw_temporal_cache(RAW, CACHE)

print(json.dumps(cache_report, indent=2))
assert cache_report['rows'] == 2_365_424

## 2. Build all three prespecified episodes

A failed episode remains failed. Do not replace it with another family.

In [ ]:
suite = build_unsw_event_suite(
    RAW,
    CACHE,
    EVENT_DIR,
    families=['DoS', 'Exploits', 'Reconnaissance'],
)
print(json.dumps(suite, indent=2))

## 3. Audit every constructed event

In [ ]:
dataset_configs = {
    'DoS': (
        'configs/datasets/nf_unsw_nb15_v3_dos_source.yaml',
        'configs/datasets/nf_unsw_nb15_v3_dos_target.yaml',
    ),
    'Exploits': (
        'configs/datasets/nf_unsw_nb15_v3_exploits_source.yaml',
        'configs/datasets/nf_unsw_nb15_v3_exploits_target.yaml',
    ),
    'Reconnaissance': (
        'configs/datasets/nf_unsw_nb15_v3_reconnaissance_source.yaml',
        'configs/datasets/nf_unsw_nb15_v3_reconnaissance_target.yaml',
    ),
}
constructed = {
    row['family'] for row in suite['outcomes'] if row['status'] == 'constructed'
}
audit_reports = {}
for family in sorted(constructed):
    source_cfg, target_cfg = dataset_configs[family]
    source_report = audit_dataset(source_cfg, Path('results/audits') / f'v019_{family}_source.json')
    target_report = audit_dataset(target_cfg, Path('results/audits') / f'v019_{family}_target.json')
    audit_reports[family] = {'source': source_report, 'target': target_report}
    print(family, 'source rows=', source_report['rows_audited'], 'target rows=', target_report['rows_audited'])

## 4. Run authoritative seed 11

Review each saved score trace and candidate audit before running the remaining seeds. Do not change candidate values.

In [ ]:
benchmark_configs = {
    'DoS': 'configs/guard_benchmarks/v019_unsw_dos.yaml',
    'Exploits': 'configs/guard_benchmarks/v019_unsw_exploits.yaml',
    'Reconnaissance': 'configs/guard_benchmarks/v019_unsw_reconnaissance.yaml',
}
seed11_summaries = {}
for family in ['DoS', 'Exploits', 'Reconnaissance']:
    if family not in constructed:
        print(family, 'skipped because event construction failed')
        continue
    summary = run_guard_benchmark(benchmark_configs[family])
    seed11_summaries[family] = summary
    print('\n', family)
    for row in summary['guard_results']:
        print(row['detector'], row['guard_status'], row['post_change_detected'], row['detection_delay_windows'])

## 5. Run the remaining paired seeds

Set `RUN_FULL_MATRICES = True` only after checking the seed-11 files.

In [ ]:
RUN_FULL_MATRICES = False

matrix_configs = {
    'DoS': 'configs/matrices/v019_unsw_dos_guards.yaml',
    'Exploits': 'configs/matrices/v019_unsw_exploits_guards.yaml',
    'Reconnaissance': 'configs/matrices/v019_unsw_reconnaissance_guards.yaml',
}
if RUN_FULL_MATRICES:
    for family in ['DoS', 'Exploits', 'Reconnaissance']:
        if family not in constructed:
            continue
        matrix = load_yaml(matrix_configs[family])
        base = load_yaml(matrix['base_benchmark'])
        for combination in itertools.product(*matrix['axes'].values()):
            override = {}
            for value in combination:
                override = deep_merge(override, value)
            summary = run_guard_benchmark(deep_merge(base, override))
            print(family, 'seed', summary['seed'], 'completed')
else:
    print('Full matrices are paused. Review seed-11 evidence first.')

## 6. Aggregate after all eligible matrices finish

In [ ]:
if RUN_FULL_MATRICES:
    aggregate_manifest = aggregate_guard_benchmarks(RESULTS_DIR, AGGREGATE_DIR)
    print(json.dumps(aggregate_manifest, indent=2))
else:
    print('Aggregation is paused until the full matrices finish.')